# Cào dữ liệu thực vật Việt Nam từ GBIF: năm 2024

Mỗi notebook cào một năm. Chạy hết các ô theo thứ tự rồi tải file CSV ở `OUT_DIR`.
Trên Kaggle: bật Internet trong Settings, kết quả nằm ở `/kaggle/working`.

In [ ]:
YEAR = 2024
PAGE = 300  # GBIF cho toi da 300 ban ghi moi trang
import os
OUT_DIR = "/kaggle/working" if "KAGGLE_KERNEL_RUN_TYPE" in os.environ else "data"

In [ ]:
import time
import pandas as pd
import requests

URL = "https://api.gbif.org/v1/occurrence/search"
BASE = dict(country="VN", kingdomKey=6, hasCoordinate="true", hasGeospatialIssue="false", year=YEAR)

FIELDS = [
    "gbifID", "scientificName", "species", "genus", "family", "order", "class", "phylum",
    "taxonRank", "iucnRedListCategory", "basisOfRecord", "datasetName", "institutionCode",
    "year", "month", "day", "decimalLatitude", "decimalLongitude",
    "coordinateUncertaintyInMeters", "elevation", "stateProvince", "recordedBy", "license",
]


def get(params):
    for attempt in range(5):
        try:
            r = requests.get(URL, params={**BASE, **params}, timeout=60)
            if r.status_code == 429:  # bi gioi han toc do
                time.sleep(10 * (attempt + 1))
                continue
            r.raise_for_status()
            return r.json()
        except requests.RequestException:
            time.sleep(3 * (attempt + 1))
    raise RuntimeError(f"GBIF khong phan hoi (offset={params.get('offset')})")


expected = get({"limit": 0})["count"]
print(f"Nam {YEAR}: GBIF bao co {expected} ban ghi")

rows = []
for offset in range(0, expected, PAGE):
    for rec in get({"limit": PAGE, "offset": offset})["results"]:
        row = {k: rec.get(k) for k in FIELDS}
        gadm = rec.get("gadm") or {}
        row["province_gadm"] = (gadm.get("level1") or {}).get("name")
        rows.append(row)
    time.sleep(0.3)
    print(f"  offset {offset}: tong {len(rows)}")

df = pd.DataFrame(rows).drop_duplicates("gbifID")
print(f"Thu duoc {len(df)} / {expected} ban ghi")

In [ ]:
import os
os.makedirs(OUT_DIR, exist_ok=True)
path = os.path.join(OUT_DIR, f"gbif_vn_plants_{YEAR}.csv")
df.to_csv(path, index=False, encoding="utf-8-sig")
print("Da luu", path)

## Kiểm tra nhanh

In [ ]:
print("So dong:", len(df), "| so cot:", df.shape[1])
print("Khop voi GBIF:", len(df) == expected, f"({len(df)} / {expected})")
print()
print("Ty le thieu theo cot:")
print(df.isna().mean().round(2).sort_values(ascending=False).head(10).to_string())
df.head()